# R11 · Diagnóstico de los datos

**Proyecto Final Integrador · Machine Learning I · Universidad EAN**
Integrantes: Claudia Naranjo, Yery Pedraza y Nicolás Ruiz.

Diagnóstico de los datos que acompaña a [`ficha_datos.md`](ficha_datos.md): si los datos están en
condiciones de sostener el modelo B y cuánto cuesta ajustarlo.

| Sección | Criterio de R11 |
|---|---|
| §1 Variable objetivo y cierres sin pago | Objetivo con su definición; desbalance |
| §2 Tipos y faltantes | Tipos, % de faltantes por columna |
| §3 Duplicados | Duplicados |
| §4 Categorías inconsistentes | Categorías inconsistentes |
| §5 Mecanismo de faltante | Mecanismo de la columna con más faltantes |
| §6 Fuga y datos sensibles | Columnas con riesgo de fuga o sensibles, dentro o fuera |
| §7 Costo de la SVM RBF | Tiempo en tres tamaños o más y filas para la búsqueda |

**Sin mirar la prueba.** Los diagnósticos que deciden transformaciones (faltantes de las
predictoras, desbalance de la muestra, curva de la SVM) se calculan solo con facturas emitidas de
2022 a 2024. Las de 2025 y 2026 quedan reservadas para la prueba final. Las tablas de tipos,
faltantes y duplicados de toda la base son descriptivas: no deciden ninguna transformación.

**Confidencialidad.** El cuaderno solo muestra cifras agregadas: nunca imprime filas de
clientes ni de facturas.

## 0. Configuración

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

# Raíz del proyecto: la primera carpeta hacia arriba que contiene datos_crudos/ (ruta relativa).
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datos_crudos").is_dir())
DATOS = RAIZ / "datos_crudos"
sys.path.insert(0, str(RAIZ / "src"))

from cartera import etiqueta as et
from cartera.config import SEMILLA  # semilla única del proyecto

np.random.seed(SEMILLA)

print("Semilla", SEMILLA, "| Python", sys.version.split()[0], "| pandas", pd.__version__,
      "| numpy", np.__version__, "| scikit-learn", sklearn.__version__)

Semilla 2026 | Python 3.14.7 | pandas 3.0.6 | numpy 2.5.3 | scikit-learn 1.9.1


In [2]:
def leer(nombre):
    df = pd.read_csv(DATOS / nombre, encoding="utf-8-sig", low_memory=False)
    derivadas = {"fecha_anio", "fecha_mes", "fecha_trimestre", "fecha_dia_mes",
                 "fecha_dia_semana", "fecha_es_fin_de_mes"}
    for c in df.columns:
        if c.startswith("fecha") and c not in derivadas:
            df[c] = pd.to_datetime(df[c])
    return df

saldos = leer("saldos_anonimizados.csv")
movs = leer("movimientos_anonimizados.csv")
print("saldos:", saldos.shape, "| movimientos:", movs.shape)

saldos: (19262, 54) | movimientos: (40519, 44)


## 1. Variable objetivo y cierres sin pago

**Pago tardío** (variable objetivo de B) vale 1 si el atraso de la factura supera 30 días. El
atraso se medía hasta que el saldo quedó en cero, pero un saldo puede llegar a cero con una nota
crédito o un ajuste, sin que entre dinero. La decisión del equipo mide el atraso hasta el último pago
con recibo de caja (RC). La regla está en `src/cartera/etiqueta.py`, para que R12, R13 y el
backtest usen la misma.

### 1.1 Base: facturas cerradas con atraso medible

Una factura cerrada tiene el saldo en cero (`SIN_SALDO`). Para medir su atraso hace falta la
fecha de cancelación, y su plazo no puede ser negativo (vencimiento antes de la emisión es un
error de registro). Esta tabla explica por qué hay 17.235 facturas cerradas, pero solo 17.209
entran en el cálculo.

In [3]:
fac = saldos[saldos.es_deuda_facturada]
todas_cerradas = fac[fac.estado_vencimiento_actual == "SIN_SALDO"]
sin_fecha = todas_cerradas.fecha_cancelacion_registrada.isna()
plazo_neg = todas_cerradas.plazo_dias_observado < 0
cerr = et.facturas_cerradas(saldos)

assert len(todas_cerradas) - (sin_fecha | plazo_neg).sum() == len(cerr)
pd.Series({
    "Facturas cerradas (saldo en cero)": len(todas_cerradas),
    "− sin fecha de cancelación": int(sin_fecha.sum()),
    "− con plazo negativo": int((plazo_neg & ~sin_fecha).sum()),
    "= Facturas cerradas con atraso medible": len(cerr),
}).to_frame("facturas")

,facturas
Facturas cerradas (saldo en cero),17235
− sin fecha de cancelación,25
− con plazo negativo,1
= Facturas cerradas con atraso medible,17209


### 1.2 Cómo se cerraron

- **RC:** el último documento que acreditó la factura fue un recibo de caja.
- **Mixta:** la cerró una nota o un ajuste, pero antes el cliente pagó con al menos un RC.
- **Sin RC:** la cerró una nota o un ajuste y nunca hubo RC.

La columna «% tardías» usa todavía la definición anterior (atraso hasta que el saldo quedó en
cero), para ver dónde está el problema.

In [4]:
cerr["tipo_cierre"] = et.tipo_cierre(cerr)
cerr["atraso_anterior"] = et.atraso_hasta_cancelacion(cerr)
cerr["tardia_anterior"] = et.pago_tardio(cerr.atraso_anterior)

orden = [et.CIERRE_RC, et.CIERRE_MIXTA, et.CIERRE_SIN_RC]
por_tipo = cerr.groupby("tipo_cierre").agg(facturas=("id_saldo", "size"),
                                           pct_tardias=("tardia_anterior", "mean")).reindex(orden)
por_tipo["% de las cerradas"] = (por_tipo.facturas / len(cerr) * 100).round(1)
por_tipo["% tardías (definición anterior)"] = (por_tipo.pct_tardias * 100).round(1)
por_tipo.drop(columns="pct_tardias")

,facturas,% de las cerradas,% tardías (definición anterior)
tipo_cierre,,,
RC,15414,89.6,33.0
mixta,511,3.0,64.4
sin RC,1284,7.5,35.4


### 1.3 Mixtas: medir hasta el último RC

En las mixtas, el atraso se mide hasta la fecha del último RC (`fecha_ultimo_recaudo`). La
cobertura es la parte del valor de la factura que se pagó con RC.

In [5]:
cerr["atraso"] = et.atraso_d16(cerr)
cerr["tardia"] = et.pago_tardio(cerr.atraso)
cerr["cobertura_rc"] = et.cobertura_rc(cerr, movs)

mix = cerr[cerr.tipo_cierre == et.CIERRE_MIXTA]
pd.Series({
    "Mixtas": len(mix),
    "% tardías, hasta que el saldo quedó en cero": round(mix.tardia_anterior.mean() * 100, 1),
    "% tardías, hasta el último RC": round(mix.tardia.mean() * 100, 1),
    "Mixtas cuya etiqueta cambia": int((mix.tardia != mix.tardia_anterior).sum()),
    "  de tardía a no tardía": int(((mix.tardia_anterior == 1) & (mix.tardia == 0)).sum()),
    "  de no tardía a tardía": int(((mix.tardia_anterior == 0) & (mix.tardia == 1)).sum()),
    "Cobertura RC, mediana (%)": round(mix.cobertura_rc.median() * 100, 2),
    "Mixtas con cobertura RC menor que 90 %": int((mix.cobertura_rc < 0.9).sum()),
}, dtype=object).to_frame("valor")

,valor
Mixtas,511
"% tardías, hasta que el saldo quedó en cero",64.4
"% tardías, hasta el último RC",31.5
Mixtas cuya etiqueta cambia,168
de tardía a no tardía,168
de no tardía a tardía,0
"Cobertura RC, mediana (%)",99.62
Mixtas con cobertura RC menor que 90 %,103


### 1.4 Etiqueta con la regla del último RC

Las cerradas sin RC quedan fuera de la etiqueta de B. El caso borde de las mixtas con poco RC
se aplica en §1.6. Esta tabla da las facturas etiquetables y
el desbalance de la etiqueta. Todavía no aplica la regla de desenlace conocido, que se
aplica en R12 en cada corte de la partición.

In [6]:
etiquetables = cerr[cerr.tardia.notna()]
assert len(etiquetables) == len(cerr) - (cerr.tipo_cierre == et.CIERRE_SIN_RC).sum()
pd.Series({
    "Facturas cerradas con atraso medible": len(cerr),
    "− sin RC (fuera de la etiqueta de B)": int((cerr.tipo_cierre == et.CIERRE_SIN_RC).sum()),
    "= Facturas con etiqueta": len(etiquetables),
    "% con pago tardío, definición anterior (todas las cerradas)": round(cerr.tardia_anterior.mean() * 100, 1),
    "% con pago tardío, etiqueta final": round(etiquetables.tardia.mean() * 100, 1),
}, dtype=object).to_frame("valor")

,valor
Facturas cerradas con atraso medible,17209
− sin RC (fuera de la etiqueta de B),1284
= Facturas con etiqueta,15925
"% con pago tardío, definición anterior (todas las cerradas)",34.1
"% con pago tardío, etiqueta final",33.0


### 1.5 Caso borde: mixtas con poco RC

Algunas mixtas tienen un RC que cubre poco del valor de la factura: el cliente pagó una parte y
el resto se cerró con una nota o un ajuste. Hay dos opciones abiertas en `PENDIENTES.md`:

- **(a)** medirlas todas hasta el último RC, como el resto de las mixtas, y declararlo;
- **(b)** exigir que el RC cubra al menos el 90 % del valor y dejar fuera de la etiqueta a las demás.

Las tablas comparan las mixtas con poco RC (cobertura menor que 90 %) con las demás mixtas y
muestran cuánto cambia la etiqueta con cada opción.

**Decisión (21/10/2026): opción (b).** La etiqueta debe describir un pago: si el RC cubre
menos del 90 %, la mayor parte de la factura se cerró sin dinero, como en las facturas sin RC. La
etiqueta final está en §1.6.

In [7]:
COBERTURA_MIN = 0.9
mix = mix.assign(poco_rc=mix.cobertura_rc < COBERTURA_MIN)

pd.cut(mix.loc[mix.poco_rc, "cobertura_rc"], [0, 0.25, 0.5, 0.75, COBERTURA_MIN],
       right=False).value_counts().sort_index().rename_axis("cobertura RC").to_frame("mixtas con poco RC")

,mixtas con poco RC
cobertura RC,
"[0.0, 0.25)",23
"[0.25, 0.5)",14
"[0.5, 0.75)",38
"[0.75, 0.9)",28


In [8]:
def resumen(g):
    return pd.Series({
        "facturas": len(g),
        "% tardías, hasta el saldo en cero": round(g.tardia_anterior.mean() * 100, 1),
        "% tardías, hasta el último RC": round(g.tardia.mean() * 100, 1),
        "etiquetas que cambian": int((g.tardia != g.tardia_anterior).sum()),
        "atraso mediano hasta el último RC (días)": g.atraso.median(),
        "clientes distintos": g.nit_cliente.nunique(),
    }, dtype=object)

pd.DataFrame({"mixtas con poco RC": resumen(mix[mix.poco_rc]),
              "mixtas con RC ≥ 90 %": resumen(mix[~mix.poco_rc])})

,mixtas con poco RC,mixtas con RC ≥ 90 %
facturas,103,408
"% tardías, hasta el saldo en cero",83.5,59.6
"% tardías, hasta el último RC",54.4,25.7
etiquetas que cambian,30,138
atraso mediano hasta el último RC (días),39.0,11.0
clientes distintos,66,165


In [9]:
# Efecto de cada opción sobre la etiqueta completa de B
opcion_a = etiquetables
opcion_b = etiquetables[~((etiquetables.tipo_cierre == et.CIERRE_MIXTA)
                          & (etiquetables.cobertura_rc < COBERTURA_MIN))]

def efecto(g):
    return pd.Series({
        "facturas con etiqueta": len(g),
        "% con pago tardío": round(g.tardia.mean() * 100, 2),
        "% del monto (total_debitos) con pago tardío":
            round(g.loc[g.tardia == 1, "total_debitos"].sum() / g.total_debitos.sum() * 100, 2),
    }, dtype=object)

pd.DataFrame({"(a) todas hasta el último RC": efecto(opcion_a),
              "(b) solo mixtas con RC ≥ 90 %": efecto(opcion_b)})

,(a) todas hasta el último RC,(b) solo mixtas con RC ≥ 90 %
facturas con etiqueta,15925,15822
% con pago tardío,32.98,32.84
% del monto (total_debitos) con pago tardío,29.04,28.75


In [10]:
poco = mix[mix.poco_rc]
pd.Series({
    "Mixtas con poco RC, % de las facturas con etiqueta": round(len(poco) / len(etiquetables) * 100, 2),
    "Mixtas con poco RC, % del monto con etiqueta":
        round(poco.total_debitos.sum() / etiquetables.total_debitos.sum() * 100, 2),
    "Mixtas con poco RC tardías hasta el último RC": int((poco.tardia == 1).sum()),
}, dtype=object).to_frame("valor")

,valor
"Mixtas con poco RC, % de las facturas con etiqueta",0.65
"Mixtas con poco RC, % del monto con etiqueta",0.95
Mixtas con poco RC tardías hasta el último RC,56


### 1.6 Etiqueta final de B

La etiqueta de B usa `et.atraso_etiqueta`: además de las cerradas sin RC, quedan fuera las mixtas
cuyos RC cubren menos del 90 % del valor. El 90 % es un criterio del equipo, no sale de los datos.
Para mostrar que la etiqueta no depende de ese número, se repite con 80 % y 95 %.

In [11]:
filas = {}
for umbral in [0.80, et.COBERTURA_MIN_RC, 0.95]:
    tardia = et.pago_tardio(et.atraso_etiqueta(cerr, movs, cobertura_min=umbral))
    con_etiqueta = cerr[tardia.notna()]
    t = tardia[tardia.notna()]
    filas[f"RC ≥ {umbral:.0%}"] = pd.Series({
        "mixtas fuera de la etiqueta": int(((cerr.tipo_cierre == et.CIERRE_MIXTA) & tardia.isna()).sum()),
        "facturas con etiqueta": len(con_etiqueta),
        "% con pago tardío": round(t.mean() * 100, 2),
        "% del monto con pago tardío":
            round(con_etiqueta.loc[t == 1, "total_debitos"].sum() / con_etiqueta.total_debitos.sum() * 100, 2),
    }, dtype=object)
pd.DataFrame(filas)

,RC ≥ 80%,RC ≥ 90%,RC ≥ 95%
mixtas fuera de la etiqueta,80,103,126
facturas con etiqueta,15845,15822,15799
% con pago tardío,32.86,32.84,32.84
% del monto con pago tardío,28.86,28.75,28.73


In [12]:
cerr["tardia_b"] = et.pago_tardio(et.atraso_etiqueta(cerr, movs))
etiqueta_b = cerr[cerr.tardia_b.notna()]
assert len(etiqueta_b) == len(opcion_b)

pd.Series({
    "Facturas cerradas con atraso medible": len(cerr),
    "− sin RC": int((cerr.tipo_cierre == et.CIERRE_SIN_RC).sum()),
    "− mixtas con RC menor que 90 %": int(((cerr.tipo_cierre == et.CIERRE_MIXTA) & cerr.tardia_b.isna()).sum()),
    "= Facturas con etiqueta de B": len(etiqueta_b),
    "  con pago tardío (1)": int((etiqueta_b.tardia_b == 1).sum()),
    "  sin pago tardío (0)": int((etiqueta_b.tardia_b == 0).sum()),
    "% con pago tardío": round(etiqueta_b.tardia_b.mean() * 100, 2),
}, dtype=object).to_frame("valor")

,valor
Facturas cerradas con atraso medible,17209
− sin RC,1284
− mixtas con RC menor que 90 %,103
= Facturas con etiqueta de B,15822
con pago tardío (1),5196
sin pago tardío (0),10626
% con pago tardío,32.84


## 2. Tipos y faltantes

Tipos de cada tabla y porcentaje de faltantes por columna. Solo se listan las columnas con algún
faltante; las demás están completas.

In [13]:
from cartera import muestra as mu

def tipos(df):
    return df.dtypes.astype(str).value_counts().rename_axis("tipo").to_frame("columnas")

pd.concat({"saldos": tipos(saldos), "movimientos": tipos(movs)}, axis=1).fillna(0).astype(int)

,saldos,movimientos
,columnas,columnas
tipo,,
str,18,20
int64,17,11
datetime64[us],9,7
float64,7,4
bool,3,2


In [14]:
def faltantes(df, nombre):
    pct = (df.isna().mean() * 100).round(2)
    pct = pct[pct > 0].sort_values(ascending=False)
    print(f"{nombre}: {len(pct)} de {df.shape[1]} columnas con faltantes")
    return pct.to_frame("% faltantes")

faltantes(saldos, "saldos")

saldos: 7 de 54 columnas con faltantes


,% faltantes
dias_mora_actual,96.27
fecha_primer_recaudo,11.50
fecha_ultimo_recaudo,11.50
dias_hasta_primer_recaudo,11.50
fecha_cancelacion_registrada,3.89
dias_hasta_cancelacion,3.89
tipo_documento_cierre,3.74


In [15]:
faltantes(movs, "movimientos")

movimientos: 1 de 44 columnas con faltantes


,% faltantes
dias_recaudo_vs_vencimiento,55.6


**Faltantes en las predictoras, solo en el periodo de estudio (2022–2024).** Las columnas con
faltantes de arriba son todas (P), posteriores a la emisión, y no entran al modelo. Esta celda
verifica que las variables provisionales tampoco tienen faltantes en la muestra de 2022–2024.

In [16]:
FECHA_CORTE = pd.Timestamp("2026-10-18")
estudio = mu.desenlace_conocido(saldos, movs, FECHA_CORTE)
estudio = estudio[(estudio.fecha >= "2022-01-01") & (estudio.fecha < "2025-01-01")]
estudio = mu.agregar_log_monto_inicial(estudio, movs)
PRED = mu.NUM_PROVISIONALES + mu.CAT_PROVISIONALES
(estudio[PRED].isna().mean() * 100).round(2).to_frame("% faltantes, 2022–2024")

,"% faltantes, 2022–2024"
plazo_dias_observado,0.0
log_monto_inicial,0.0
fecha_mes,0.0
empresa,0.0
unidad_negocio,0.0


## 3. Duplicados

In [17]:
def duplicados(df, llave):
    sin_ids = df.drop(columns=[c for c in df.columns if c.startswith("id_")])
    return pd.Series({
        "Llave única (id_saldo / id_movimiento_saldo)": df[llave].is_unique,
        "Filas duplicadas completas": int(df.duplicated().sum()),
        "Filas duplicadas sin columnas id_*": int(sin_ids.duplicated().sum()),
    }, dtype=object)

pd.DataFrame({"saldos": duplicados(saldos, "id_saldo"),
              "movimientos": duplicados(movs, "id_movimiento_saldo")})

,saldos,movimientos
Llave única (id_saldo / id_movimiento_saldo),True,True
Filas duplicadas completas,0,0
Filas duplicadas sin columnas id_*,0,0


## 4. Categorías inconsistentes

**Escritura.** Para cada columna de texto con pocas categorías, se compara el número de categorías
con el que queda al quitar espacios y unificar mayúsculas. Si baja, hay variantes de una misma
categoría.

In [18]:
texto = [c for c in saldos.columns if saldos[c].dtype == object or str(saldos[c].dtype) in ("str", "string")]
filas = []
for c in texto:
    v = saldos[c].dropna().astype(str)
    if v.nunique() <= 60:
        filas.append({"columna": c, "categorías": v.nunique(),
                      "tras normalizar": v.str.strip().str.upper().nunique(),
                      "valores con espacios sobrantes": int((v != v.str.strip()).sum())})
pd.DataFrame(filas).set_index("columna")

,categorías,tras normalizar,valores con espacios sobrantes
columna,,,
id_compania,4,4,0
empresa,4,4,0
id_unidad_negocio,11,11,0
unidad_negocio,11,11,0
tipo_cliente,3,3,0
actividad_economica_sintetica_desc,41,41,0
id_vendedor,46,46,0
nombre_vendedor,22,22,0
tipo_documento_origen,19,19,0


**Contenido.** Las inconsistencias no están en la escritura sino en lo que dicen algunas columnas:

In [19]:
fac = saldos[saldos.es_deuda_facturada]
c15 = saldos.condicion_pago_inferida == "C15"
pd.Series({
    "Saldos con condición C15 y plazo negativo": int((c15 & (saldos.plazo_dias_observado < 0)).sum()),
    "Facturas con saldo en cero (SIN_SALDO) sin fecha de cancelación": int(((fac.estado_vencimiento_actual == "SIN_SALDO") & fac.fecha_cancelacion_registrada.isna()).sum()),
    "Valores distintos de cuota (el diccionario dice 'normalmente 1')": sorted(saldos.cuota.unique().tolist()),
    "¿Aparece la cuenta 131510 del diccionario?": bool((saldos.codigo_cuenta == 131510).any()),
    "¿Aparece el tipo de cliente INDETERMINADO del diccionario?": bool((saldos.tipo_cliente == "INDETERMINADO").any()),
}, dtype=object).to_frame("valor")

,valor
Saldos con condición C15 y plazo negativo,245
Facturas con saldo en cero (SIN_SALDO) sin fecha de cancelación,25
Valores distintos de cuota (el diccionario dice 'normalmente 1'),[0]
¿Aparece la cuenta 131510 del diccionario?,False
¿Aparece el tipo de cliente INDETERMINADO del diccionario?,False


In [20]:
saldos.groupby("condicion_pago_inferida").plazo_dias_observado.agg(["size", "min", "median", "max"]).rename(
    columns={"size": "saldos", "min": "plazo mínimo", "median": "plazo mediano", "max": "plazo máximo"})

,saldos,plazo mínimo,plazo mediano,plazo máximo
condicion_pago_inferida,,,,
>C90,1,317,317.0,317
C15,16707,-1563,15.0,17
C30,1785,18,30.0,32
C45,317,33,46.0,47
C60,446,59,61.0,62
C90,6,63,63.0,92


**Lectura.**
- No hay variantes de escritura.
- `condicion_pago_inferida` se infiere del plazo por tramos, así que repite la información de
  `plazo_dias_observado`. Además, el tramo C15 incluye los plazos negativos, que son errores de
  registro. No entra como predictora: se usa el plazo.
- Las 25 facturas en cero sin fecha de cancelación no tienen atraso medible y quedan fuera de la
  etiqueta (§1.1).
- `cuota` vale 0 en todas las filas y no se usa.
- Las dos categorías del diccionario que no aparecen se documentan en la ficha.

**Desbalance del objetivo.** En toda la etiqueta, 32,84 % de pago tardío (§1.6). En el periodo de
estudio:

In [21]:
estudio.groupby(estudio.fecha.dt.year).tardia.agg(facturas="size", pct_tardias="mean").assign(
    pct_tardias=lambda t: (t.pct_tardias * 100).round(1)).rename_axis("año de emisión")

,facturas,pct_tardias
año de emisión,,
2022,2325,29.8
2023,2617,30.2
2024,2820,30.2


Alrededor de una de cada tres facturas es tardía: el desbalance es moderado. Se trata dando más
peso a las tardías solo en entrenamiento (`diseno_experimental.md` §6.5), y se evalúa con AUC-PR, no con
exactitud.

## 5. Mecanismo de faltante de la columna con más faltantes

La columna con más faltantes es `dias_mora_actual` (96,27 % de los saldos). La tabla cruza el
faltante con el estado del saldo a la fecha de corte.

In [22]:
pd.crosstab(saldos.estado_vencimiento_actual, saldos.dias_mora_actual.isna().map({True: "falta", False: "tiene valor"}))

dias_mora_actual,falta,tiene valor
estado_vencimiento_actual,,
CORRIENTE,0,117
SALDO_A_FAVOR,2,0
SIN_SALDO,18542,0
VENCIDA,0,601


**Mecanismo asumido: faltante estructural, que depende de una variable observada (MAR).** Falta
exactamente cuando el saldo está en cero o a favor: para esos saldos los «días de mora» no aplican.
El faltante se explica por completo con `estado_vencimiento_actual`, sin depender del valor que
falta. No se imputa: la columna es (P), describe el estado al corte y no entra al modelo.

## 6. Columnas con riesgo de fuga o datos sensibles

Todo el conjunto está anonimizado por la compañía: ninguna columna identifica a una persona, así que
no hay atributos sensibles. El riesgo que queda es la fuga.

| Columna | Riesgo | ¿Predictora? | Por qué |
|---|---|---|---|
| `dias_hasta_cancelacion` | Fuga | **Fuera** | Es `fecha_cancelacion − emisión`: con el plazo, da el atraso, que es la etiqueta |
| `porcentaje_pagado`, `num_recaudos_rc`, `fecha_ultimo_recaudo`, `tipo_documento_cierre` y demás (P) | Fuga | **Fuera** | Se conocen después de la emisión; describen cómo terminó la factura |
| `total_debitos` | Fuga leve | **Fuera** | Acumulado que puede crecer después de la emisión; se usa el débito del primer movimiento |
| `codigo_cuenta` | Fuga | **Fuera** | La cuenta 131080 (difícil cobro) es una reclasificación posterior; en las facturas no aparece, pero no se arriesga |
| `nit_cliente`, `cliente` y demás identificadores | Memorización | **Fuera** | El modelo aprendería quién es el cliente, no cómo paga |
| `nombre_vendedor` | No es sensible: está anonimizado | Se decide en R12 | No es fuga: se conoce al emitir. Si entra, se revisa en R12 junto con la lista de variables |
| `tipo_cliente` | No es sensible: está anonimizado | Se decide en R12 | No es fuga, pero es inferido, no oficial, y el 98,3 % es una sola categoría: aporta poca información |
| `actividad_economica_sintetica` | Inventada | **Fuera** | Se generó al azar al anonimizar |

La comprobación de que dejar fuera `dias_hasta_cancelacion` es necesario:

In [23]:
cerr_cancel = et.facturas_cerradas(saldos)
atraso_reconstruido = cerr_cancel.dias_hasta_cancelacion - cerr_cancel.plazo_dias_observado
pd.Series({
    "Cerradas con atraso medible": len(cerr_cancel),
    "% en que dias_hasta_cancelacion − plazo = atraso hasta el saldo en cero":
        round((atraso_reconstruido == et.atraso_hasta_cancelacion(cerr_cancel)).mean() * 100, 1),
}, dtype=object).to_frame("valor")

,valor
Cerradas con atraso medible,17209
% en que dias_hasta_cancelacion − plazo = atraso hasta el saldo en cero,100.0


## 7. Costo de la SVM con núcleo RBF

**Qué se mide.** El tiempo de ajuste de una SVM RBF (`C = 1`, `gamma = "scale"`,
`class_weight = "balanced"`) con muestras de entrenamiento de distinto tamaño, y su AUC-PR en
validación. Misma muestra, partición y variables que el modelo de referencia de R10: entrenamiento
2022–2023 y validación 2024, con la escala y la codificación dentro del `Pipeline`. Cada tiempo es la
mediana de 3 ajustes. Se usa `decision_function` y no `probability=True`, que agrega una validación
cruzada interna de cinco pliegues y multiplica el tiempo.

Como comparación, una regresión logística con todo el entrenamiento.

In [24]:
import time

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

ent = estudio[estudio.fecha < "2024-01-01"]
val = estudio[estudio.fecha >= "2024-01-01"]
assert ent.fecha.max() < val.fecha.min()

def pipeline(estimador):
    prep = ColumnTransformer([
        ("num", StandardScaler(), mu.NUM_PROVISIONALES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), mu.CAT_PROVISIONALES),
    ])
    return Pipeline([("prep", prep), ("est", estimador)])

def medir(estimador, datos, repeticiones=3):
    tiempos = []
    for _ in range(repeticiones):
        modelo = pipeline(estimador)
        inicio = time.perf_counter()
        modelo.fit(datos[PRED], datos.tardia)
        tiempos.append(time.perf_counter() - inicio)
    puntaje = modelo.decision_function(val[PRED])
    return np.median(tiempos), average_precision_score(val.tardia, puntaje)

filas = []
for n in [500, 1000, 2000, 4000, len(ent)]:
    sub = ent.sample(n=n, random_state=SEMILLA) if n < len(ent) else ent
    t, ap = medir(SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced"), sub)
    filas.append({"modelo": "SVM RBF", "filas de entrenamiento": n, "segundos de ajuste": round(t, 3), "AUC-PR validación": round(ap, 3)})
t, ap = medir(LogisticRegression(class_weight="balanced", max_iter=1000), ent)
filas.append({"modelo": "Regresión logística", "filas de entrenamiento": len(ent), "segundos de ajuste": round(t, 3), "AUC-PR validación": round(ap, 3)})
curva = pd.DataFrame(filas)
curva

,modelo,filas de entrenamiento,segundos de ajuste,AUC-PR validación
0,SVM RBF,500,0.005,0.364
1,SVM RBF,1000,0.011,0.357
2,SVM RBF,2000,0.039,0.354
3,SVM RBF,4000,0.192,0.353
4,SVM RBF,4942,0.252,0.359
5,Regresión logística,4942,0.009,0.353


**Extrapolación.** Con los tiempos medidos de la SVM se ajusta una curva `tiempo = a · filasᵇ` (una
recta en escala logarítmica). `b` dice qué tan rápido crece el costo: 1 sería lineal. Con ella se
estima el tiempo de un ajuste con todo el entrenamiento posible y el de una búsqueda de
hiperparámetros de 30 combinaciones con validación cruzada de 5 pliegues (150 ajustes).

In [25]:
svm = curva[curva.modelo == "SVM RBF"]
b, log_a = np.polyfit(np.log(svm["filas de entrenamiento"]), np.log(svm["segundos de ajuste"]), 1)

# Tamaño de referencia: todas las facturas con desenlace conocido desde 2022 (solo se cuentan)
todas_desde_2022 = mu.desenlace_conocido(saldos, movs, FECHA_CORTE)
n_max = int((todas_desde_2022.fecha >= "2022-01-01").sum())

def segundos(n):
    return float(np.exp(log_a) * n ** b)

pd.Series({
    "Exponente b (1 = crece lineal)": round(b, 2),
    "Facturas con desenlace conocido desde 2022 (tope del entrenamiento)": n_max,
    "Segundos estimados de un ajuste con ese tope": round(segundos(n_max), 1),
    "Minutos estimados de una búsqueda de 150 ajustes con ese tope": round(150 * segundos(n_max) / 60, 1),
}, dtype=object).to_frame("valor")

,valor
Exponente b (1 = crece lineal),1.79
Facturas con desenlace conocido desde 2022 (tope del entrenamiento),13212
Segundos estimados de un ajuste con ese tope,1.4
Minutos estimados de una búsqueda de 150 ajustes con ese tope,3.5


**Decisión: la búsqueda de hiperparámetros usa todas las filas de entrenamiento, sin
submuestrear.** Con el tope de 13.212 facturas, una búsqueda de 150 ajustes toma unos minutos;
submuestrear no ahorra nada que importe y dejaría fuera datos para cuando R12 agregue el historial
del cliente. Si las variables de R12 hacen la SVM mucho más lenta, la curva se repite.

Lo que muestra la curva: la AUC-PR en validación casi no cambia con el tamaño de la muestra con
estas variables, y el tiempo crece más rápido que lineal pero sigue siendo de segundos por ajuste.